# 01 — Arquitectura y servicios

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

| Grupo benchmark | Instancia | Diferenciación esperada |
|-----------------|-----------|-------------------------|
| 3D-BPP | `showcase_3d_bpp_instance.json` | best_fit ~27 emp. vs extreme_points ~22 |
| Híbrido (misma instancia 3D) | `showcase_3d_bpp_instance.json` | compaction/LS mueven 8–11 piezas |
| Mejora 3D (misma instancia 3D) | `showcase_3d_bpp_instance.json` | relocation/swap/orientation/bin_reduction refinan layout |
| Container Loading | `showcase_container_loading_instance.json` | weight_aware ~27 vs single_container ~17 |
| Single Container | `showcase_single_container_instance.json` | best_fit ~13 (94% util) vs first_fit ~16 |
| Cartonization | `showcase_cartonization_instance.json` | single-box vs multi_box |
| Palletization | `showcase_palletization_instance.json` | layer_based vs stack_based |
| Stacking-aware | `showcase_stacking_aware_instance.json` | stacking_aware vs stack_based |

Cada algoritmo implementado es un **servicio** con endpoint propio y contrato JSON homogéneo.


In [ ]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


In [ ]:
from IPython.display import Markdown, display as ipy_display
from packing_services.services.metadata_service import MetadataService
from packing_services.benchmark.profiles import list_profiles

meta = MetadataService().get_metadata()
ipy_display(Markdown(f'## **{meta.service_name}** (v{meta.service_version})\n\n{meta.description}'))
for group in ('3D_BPP', '3D_HYBRID', 'CONTAINER_LOADING', 'CARTONIZATION', 'SINGLE_CONTAINER_LOADING'):
    display.show_comparable_algorithms(group)


In [ ]:
import pandas as pd
from IPython.display import display as ipy_display
from packing_services.benchmark.profiles import list_profiles

endpoints = [
    ('Ejecutar algoritmo', 'POST /algorithms/{algorithm_name}/execute', 'Input normalizado por tipo; nombre en URL (A3)'),
    ('3D Bin Packing (legacy)', 'POST /pack/3d-bpp', 'Empaquetar en contenedores'),
    ('Benchmark', 'POST /benchmark', 'Comparar motores (perfil o engines explícitos)'),
    ('Perfiles benchmark', 'GET /benchmark/profiles', 'Catálogo de perfiles estándar'),
    ('Container Loading (legacy)', 'POST /pack/container-loading', 'Carga multi-contenedor + peso'),
    ('Cartonization (legacy)', 'POST /pack/cartonization', 'Elegir caja + layout'),
    ('Validation', 'POST /validate', 'Juez independiente'),
    ('Dataspace', 'GET /services', 'Descriptores publicables'),
]
ipy_display(pd.DataFrame(endpoints, columns=['Servicio', 'Endpoint', 'Rol']).style.hide(axis='index'))
profiles = list_profiles()
print(f'Perfiles de benchmark disponibles: {len(profiles)}')
ipy_display(pd.DataFrame([{'Tipo': p['problem_type'], 'Perfil': p['profile'], 'Motores': p['engines_count']} for p in profiles]).style.hide(axis='index'))
from _shared.loaders import SHOWCASE_DIFFERENTIATION, SHOWCASE_INSTANCES, BENCHMARK_GROUPS
showcase_rows = [{'Grupo': g, 'Instancia': SHOWCASE_INSTANCES.get(g, '—'), 'Diferenciación': SHOWCASE_DIFFERENTIATION.get(g, '—')} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(showcase_rows).style.hide(axis='index'))


In [ ]:
from _shared.loaders import build_algorithm_execute_input

# Ejemplo: un algoritmo por URL, body sin campo "algorithm"
algo = 'best_fit_decreasing_3d'
payload = build_algorithm_execute_input(algo, problem_type='3D_BPP')
print(f'POST /api/v1/algorithms/{algo}/execute')
print('Campos del body:', sorted(payload.keys()))
result = runners.run_algorithm_execute(algo, payload)
display.show_execute_result(result)


**Siguiente:** demo visual 3D-BPP → `02_demo_3d_bin_packing.ipynb`
